# 1. Setup y Carga de Datos

In [ ]:
import os
import subprocess
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.lines import Line2D

PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")

MOB_MADURO = 15
N_DESTACAR = 5
COLOR_2024 = "#2171b5"
COLOR_2025 = "#e6550d"
COLOR_2026 = "#888888"
COLOR_SINT = "#2ca02c"  # Verde para sinteticas
ALPHA_INDIVIDUAL = 0.18
ALPHA_HIGHLIGHT = 0.85

COLOR_NIVEL = {"Bajo": "#2ca02c", "Medio": "#ff7f0e", "Alto": "#d62728"}
NOMBRE_TIPOOPE = {"CC": "CC", "PP": "PP"}
NOMBRE_CLIENTENUEVO = {"V": "Nuevo", "F": "Existente"}

print(f"Raiz del proyecto: {PROJECT_ROOT}")
print(f"MOB maduro: {MOB_MADURO}")

In [ ]:
scripts = [
    "consolidar_vintage_niveles.py",
    "generar_matriz_vintage_niveles.py",
    "generar_proyeccion_chainladder_niveles.py",
    "generar_cohortes_sinteticas_niveles.py",
]
for script in scripts:
    print(f"--- {script} ---")
    result = subprocess.run(
        ["py", os.path.join(SRC_DIR, script)],
        capture_output=True, text=True, cwd=PROJECT_ROOT
    )
    # Mostrar solo las ultimas lineas
    lines = result.stdout.strip().split('\n')
    for line in lines[-5:]:
        print(f"  {line}")
    if result.returncode != 0:
        print("ERROR:", result.stderr[-500:])
        break
    print()

In [ ]:
# Leer datos
matrices_obs = pd.read_csv(os.path.join(PROCESSED_DIR, "matrices_vintage_niveles.csv"), sep=";", decimal=",")
sinteticas_df = pd.read_csv(os.path.join(PROCESSED_DIR, "sinteticas_niveles.csv"), sep=";", decimal=",")
regresion_df = pd.read_csv(os.path.join(PROCESSED_DIR, "regresion_mob1_niveles.csv"), sep=";", decimal=",")
general_sint = pd.read_csv(os.path.join(PROCESSED_DIR, "sinteticas_general_niveles.csv"), sep=";", decimal=",", index_col=0)
df_consolidado = pd.read_csv(os.path.join(PROCESSED_DIR, "vintage_niveles_consolidado.csv"), sep=";", decimal=",")

segmentos = sorted(matrices_obs["segmento"].unique())

# Ultima cohorte historica comun (la que tiene dato en la mayoria de segmentos)
ultima_hist = sorted(matrices_obs["cohorte"].unique())[-1]

# Cohortes sinteticas verdaderamente futuras (posteriores a ultima historica comun)
cohortes_sint_futuras = sorted([c for c in sinteticas_df["cohorte"].unique() if c > ultima_hist])
# Para la general, filtrar solo las futuras
general_futuras = general_sint.loc[general_sint.index > ultima_hist].copy()

print(f"Segmentos: {len(segmentos)}")
print(f"Ultima cohorte historica: {ultima_hist}")
print(f"Cohortes sinteticas futuras: {cohortes_sint_futuras}")
print(f"General futuras: {list(general_futuras.index)}")

# 2. Búsqueda del Mejor Modelo (Grid Search)

In [ ]:
combos = [("CC", "F"), ("CC", "V"), ("PP", "F"), ("PP", "V")]
niveles = ["Bajo", "Medio", "Alto"]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Estimacion MOB 1 por Nivel de Riesgo (WMA)",
             fontsize=14, fontweight="bold", y=0.98)

for idx, (tipoope, clientenuevo) in enumerate(combos):
    ax = axes[idx // 2, idx % 2]
    tipo_n = NOMBRE_TIPOOPE[tipoope]
    cliente_n = NOMBRE_CLIENTENUEVO[clientenuevo]

    for nivel in niveles:
        seg = f"{tipoope}_{clientenuevo}_{nivel}"
        sub_reg = regresion_df[regresion_df["segmento"] == seg].copy()
        if sub_reg.empty:
            continue

        hist = sub_reg[sub_reg["tipo"] == "historica"]
        sint = sub_reg[sub_reg["tipo"] == "sintetica"]

        # Observado
        obs = hist.dropna(subset=["mob1_observado"])
        ax.plot(range(len(obs)), obs["mob1_observado"].values,
                color=COLOR_NIVEL[nivel], alpha=0.6, linewidth=1.5,
                marker="o", markersize=3, label=f"{nivel} (obs)")

        # WMA sinteticas
        if not sint.empty and sint["mob1_wma"].notna().any():
            x_sint = range(len(obs), len(obs) + len(sint))
            ax.plot(x_sint, sint["mob1_wma"].values,
                    color=COLOR_NIVEL[nivel], linewidth=2, linestyle="--",
                    marker="s", markersize=5)
            # Bandas
            if sint["mob1_wma_sup"].notna().any():
                ax.fill_between(x_sint, sint["mob1_wma_inf"].values,
                                sint["mob1_wma_sup"].values,
                                color=COLOR_NIVEL[nivel], alpha=0.1)

    ax.set_title(f"{tipo_n} - {cliente_n}", fontsize=12, fontweight="bold")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    ax.grid(True, alpha=0.25, linestyle="--")
    ax.set_xlabel("Cohorte (indice)", fontsize=9)
    ax.set_ylabel("MOB 1", fontsize=9)
    ax.legend(fontsize=8, loc="best")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(REPORTS_DIR, "mob1_estimacion_niveles.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Guardado en reports/mob1_estimacion_niveles.png")

In [ ]:
# ─── PARÁMETROS CONFIGURABLES ─────────────────────────────────────────────
PARAMS_REGRESION = dict(n_cohortes=10)    # cohortes usadas para ajustar la recta
PARAMS_WMA       = dict(n_ventana=10)      # ventana de la media móvil ponderada
PARAMS_ARIMA     = dict(order=(2,1,1), n_arima=12)  # orden ARIMA y ventana histórica
N_HOLDOUT        = 3                      # cohortes retenidas para medir el error
SEGMENTOS_SKIP   = ["CC_V_Medio", "CC_V_Alto", "PP_V_Medio", "PP_V_Alto"]
# ──────────────────────────────────────────────────────────────────────────

from statsmodels.tsa.arima.model import ARIMA as _ARIMA

def _pred_regresion(vals, n_cohortes, n_steps):
    v = vals[-min(n_cohortes, len(vals)):]
    x = np.arange(len(v))
    coefs = np.polyfit(x, v, 1)
    p = np.poly1d(coefs)
    return np.array([p(len(v) - 1 + j) for j in range(1, n_steps + 1)])

def _pred_wma(vals, n_ventana, n_steps):
    v = vals[-min(n_ventana, len(vals)):]
    w = np.exp(np.linspace(0, 2, len(v))); w /= w.sum()
    wma = np.average(v, weights=w)
    diffs = np.diff(v)
    if len(diffs) > 0:
        wd = np.exp(np.linspace(0, 2, len(diffs))); wd /= wd.sum()
        tend = np.average(diffs, weights=wd)
    else:
        tend = 0.0
    return np.array([wma + tend * j for j in range(1, n_steps + 1)])

def _pred_arima(vals, n_arima, n_steps, order):
    v = vals[-min(n_arima, len(vals)):]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            res = _ARIMA(v, order=order).fit()
            return np.array(res.forecast(steps=n_steps))
        except Exception:
            return _pred_wma(vals, len(v), n_steps)

def _mae(pred, real):  return np.mean(np.abs(pred - real))
def _rmse(pred, real): return np.sqrt(np.mean((pred - real) ** 2))

# ─── Walk-forward por segmento ────────────────────────────────────────────
rows = []
for seg in sorted(matrices_obs["segmento"].unique()):
    if seg in SEGMENTOS_SKIP:
        continue
    sub = matrices_obs[matrices_obs["segmento"] == seg]
    if "MOB_1" not in sub.columns:
        continue
    mob1 = sub.set_index("cohorte")["MOB_1"].dropna().sort_index()
    vals = mob1.values.astype(float)
    if len(vals) < N_HOLDOUT + 4:
        continue
    train, test = vals[:-N_HOLDOUT], vals[-N_HOLDOUT:]

    pr = _pred_regresion(train, **PARAMS_REGRESION, n_steps=N_HOLDOUT)
    pw = _pred_wma(train,       **PARAMS_WMA,       n_steps=N_HOLDOUT)
    pa = _pred_arima(train,     **PARAMS_ARIMA,     n_steps=N_HOLDOUT)

    mejor_mae  = min([("Regresion", _mae(pr, test)),
                      ("WMA",       _mae(pw, test)),
                      ("ARIMA",     _mae(pa, test))], key=lambda x: x[1])[0]
    mejor_rmse = min([("Regresion", _rmse(pr, test)),
                      ("WMA",       _rmse(pw, test)),
                      ("ARIMA",     _rmse(pa, test))], key=lambda x: x[1])[0]

    rows.append(dict(
        segmento=seg, n_obs=len(vals),
        MAE_Reg=_mae(pr, test),    RMSE_Reg=_rmse(pr, test),
        MAE_WMA=_mae(pw, test),    RMSE_WMA=_rmse(pw, test),
        MAE_ARIMA=_mae(pa, test),  RMSE_ARIMA=_rmse(pa, test),
        Mejor_MAE=mejor_mae,       Mejor_RMSE=mejor_rmse,
    ))

df_comp = pd.DataFrame(rows).set_index("segmento")

# ─── Highlight: verde=mejor MAE, azul=mejor RMSE ─────────────────────────
def _highlight(df):
    st = pd.DataFrame("", index=df.index, columns=df.columns)
    mae_cols  = ["MAE_Reg", "MAE_WMA", "MAE_ARIMA"]
    rmse_cols = ["RMSE_Reg", "RMSE_WMA", "RMSE_ARIMA"]
    for idx in df.index:
        mc = df.loc[idx, mae_cols].idxmin()
        st.loc[idx, mc] = "background-color:#d4edda;font-weight:bold"
        rc = df.loc[idx, rmse_cols].idxmin()
        st.loc[idx, rc] = "background-color:#cce5ff;font-weight:bold"
    return st

print(
    f"Hold-out: últimas {N_HOLDOUT} cohortes  |  "
    f"Regresión(n={PARAMS_REGRESION['n_cohortes']})  "
    f"WMA(n={PARAMS_WMA['n_ventana']})  "
    f"ARIMA{PARAMS_ARIMA['order']}(n={PARAMS_ARIMA['n_arima']})"
)

cols_tabla = ["n_obs", "MAE_Reg", "MAE_WMA", "MAE_ARIMA",
              "RMSE_Reg", "RMSE_WMA", "RMSE_ARIMA", "Mejor_MAE", "Mejor_RMSE"]
fmt = {c: "{:.4f}" for c in cols_tabla if c not in ("n_obs", "Mejor_MAE", "Mejor_RMSE")}
fmt["n_obs"] = "{:.0f}"
display(
    df_comp[cols_tabla].style
    .apply(_highlight, axis=None)
    .format(fmt)
    .set_caption("Verde = mejor MAE | Azul = mejor RMSE")
)

print("\nGanadores MAE:")
print(df_comp["Mejor_MAE"].value_counts().to_string())
print("\nGanadores RMSE:")
print(df_comp["Mejor_RMSE"].value_counts().to_string())

# ─── Gráfico de barras MAE y RMSE ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(
    f"Comparación de métodos — Hold-out {N_HOLDOUT} cohortes  "
    f"(Reg n={PARAMS_REGRESION['n_cohortes']} | WMA n={PARAMS_WMA['n_ventana']} | "
    f"ARIMA{PARAMS_ARIMA['order']} n={PARAMS_ARIMA['n_arima']})",
    fontsize=11
)

for ax, metrica, col_map in zip(
    axes,
    ["MAE", "RMSE"],
    [("MAE_Reg", "MAE_WMA", "MAE_ARIMA"), ("RMSE_Reg", "RMSE_WMA", "RMSE_ARIMA")]
):
    x = np.arange(len(df_comp))
    w = 0.25
    colores = ["#4878d0", "#ee854a", "#6acc65"]
    labels  = ["Regresión", "WMA", "ARIMA"]
    for i, (col, color, lbl) in enumerate(zip(col_map, colores, labels)):
        ax.bar(x + i * w, df_comp[col], w, label=lbl, color=color, alpha=0.85)
    ax.set_xticks(x + w)
    ax.set_xticklabels(df_comp.index, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel(metrica)
    ax.set_title(f"{metrica} por segmento")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ─── GRID SEARCH DE PARÁMETROS ────────────────────────────────────────────
# Itera distintas combinaciones y muestra el MAE promedio entre segmentos.
# Cambiá los rangos para explorar más opciones.

GRID_WMA_VENTANA  = [3, 4, 5, 6, 8, 10]
GRID_REG_COHORTES = [6, 8, 10, 12, 15, 18]
GRID_ARIMA_ORDERS = [(1,0,0),(0,1,1),(1,1,0),(1,1,1),(2,1,0),(1,1,2),(2,1,1)]
N_HOLDOUT_GRID    = 3
# ──────────────────────────────────────────────────────────────────────────

# Cachear series por segmento
_cache = {}
for seg in sorted(matrices_obs["segmento"].unique()):
    if seg in SEGMENTOS_SKIP:
        continue
    sub = matrices_obs[matrices_obs["segmento"] == seg]
    if "MOB_1" not in sub.columns:
        continue
    mob1 = sub.set_index("cohorte")["MOB_1"].dropna().sort_index()
    vals = mob1.values.astype(float)
    if len(vals) >= N_HOLDOUT_GRID + 4:
        _cache[seg] = vals

def _eval_grid(fn, params_extra):
    maes = []
    for seg, vals in _cache.items():
        train, test = vals[:-N_HOLDOUT_GRID], vals[-N_HOLDOUT_GRID:]
        try:
            pred = fn(train, n_steps=N_HOLDOUT_GRID, **params_extra)
            maes.append(_mae(pred, test))
        except Exception:
            pass
    return np.mean(maes) if maes else np.nan

# ─── WMA ─────────────────────────────────────────────────────────────────
print("WMA — MAE promedio por n_ventana:")
grid_wma = [{"n_ventana": nv, "MAE_avg": _eval_grid(_pred_wma, dict(n_ventana=nv))}
            for nv in GRID_WMA_VENTANA]
df_gw = pd.DataFrame(grid_wma).set_index("n_ventana")
display(df_gw.style.format("{:.4f}").highlight_min(color="#d4edda"))
print(f"  → Mejor n_ventana: {df_gw['MAE_avg'].idxmin()}  "
      f"(MAE={df_gw['MAE_avg'].min():.4f})")

# ─── Regresión ───────────────────────────────────────────────────────────
print("\nRegresión — MAE promedio por n_cohortes:")
grid_reg = [{"n_cohortes": nc, "MAE_avg": _eval_grid(_pred_regresion, dict(n_cohortes=nc))}
            for nc in GRID_REG_COHORTES]
df_gr = pd.DataFrame(grid_reg).set_index("n_cohortes")
display(df_gr.style.format("{:.4f}").highlight_min(color="#d4edda"))
print(f"  → Mejor n_cohortes: {df_gr['MAE_avg'].idxmin()}  "
      f"(MAE={df_gr['MAE_avg'].min():.4f})")

# ─── ARIMA ───────────────────────────────────────────────────────────────
print("\nARIMA — MAE promedio por order (p,d,q):")
grid_arima = [{"order": str(o), "MAE_avg": _eval_grid(_pred_arima, dict(order=o, n_arima=12))}
              for o in GRID_ARIMA_ORDERS]
df_ga = pd.DataFrame(grid_arima).set_index("order")
display(df_ga.style.format("{:.4f}").highlight_min(color="#d4edda"))
print(f"  → Mejor order: {df_ga['MAE_avg'].idxmin()}  "
      f"(MAE={df_ga['MAE_avg'].min():.4f})")

# ─── Gráfico de líneas: evolución del MAE por parámetro ──────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle("Grid search — MAE promedio entre segmentos", fontsize=12)

axes[0].plot(df_gw.index, df_gw["MAE_avg"], marker="o", color="#ee854a")
axes[0].set_title("WMA — n_ventana"); axes[0].set_xlabel("n_ventana")
axes[0].set_ylabel("MAE avg"); axes[0].grid(alpha=0.3)
axes[0].axvline(df_gw["MAE_avg"].idxmin(), color="green", linestyle="--", alpha=0.7)

axes[1].plot(df_gr.index, df_gr["MAE_avg"], marker="o", color="#4878d0")
axes[1].set_title("Regresión — n_cohortes"); axes[1].set_xlabel("n_cohortes")
axes[1].set_ylabel("MAE avg"); axes[1].grid(alpha=0.3)
axes[1].axvline(df_gr["MAE_avg"].idxmin(), color="green", linestyle="--", alpha=0.7)

axes[2].bar(range(len(df_ga)), df_ga["MAE_avg"], color="#6acc65", alpha=0.85)
axes[2].set_xticks(range(len(df_ga)))
axes[2].set_xticklabels(df_ga.index, rotation=30, ha="right", fontsize=8)
axes[2].set_title("ARIMA — order"); axes[2].set_ylabel("MAE avg")
axes[2].grid(axis="y", alpha=0.3)
best_arima_idx = int(df_ga["MAE_avg"].values.argmin())
axes[2].bar(best_arima_idx, df_ga["MAE_avg"].iloc[best_arima_idx],
            color="green", alpha=0.9, label="mejor")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

# ─── Resumen final ────────────────────────────────────────────────────────
best_wma   = df_gw["MAE_avg"].idxmin()
best_reg   = df_gr["MAE_avg"].idxmin()
best_arima = df_ga["MAE_avg"].idxmin()
print("\n" + "=" * 55)
print("MEJORES PARÁMETROS POR MÉTODO (MAE promedio entre segmentos)")
print("=" * 55)
print(f"  Regresión  → n_cohortes = {best_reg:<4}  MAE = {df_gr['MAE_avg'].min():.4f}")
print(f"  WMA        → n_ventana  = {best_wma:<4}  MAE = {df_gw['MAE_avg'].min():.4f}")
print(f"  ARIMA      → order      = {best_arima:<12}  MAE = {df_ga['MAE_avg'].min():.4f}")
print("=" * 55)
ganador = min(
    [("Regresión", df_gr["MAE_avg"].min()),
     ("WMA",       df_gw["MAE_avg"].min()),
     ("ARIMA",     df_ga["MAE_avg"].min())],
    key=lambda x: x[1]
)
print(f"  Ganador global → {ganador[0]}  (MAE={ganador[1]:.4f})")
print("  → Para aplicarlo: cambiá METODO_SINTETICAS en generar_cohortes_sinteticas_niveles.py")


# ─── Exportar mejores parámetros para la celda de aplicación ──────────────
# Estos se usan automáticamente en la celda siguiente
MEJOR_METODO = ganador[0].lower()
if MEJOR_METODO == "regresión":
    MEJOR_METODO = "regresion"
MEJOR_WMA_VENTANA     = int(best_wma)
MEJOR_REG_COHORTES    = int(best_reg)
MEJOR_ARIMA_ORDER     = eval(best_arima) if isinstance(best_arima, str) else best_arima

print(f"\n{'='*55}")
print(f"✅ Parámetros óptimos exportados para aplicación automática:")
print(f"   MEJOR_METODO        = {MEJOR_METODO}")
print(f"   MEJOR_WMA_VENTANA   = {MEJOR_WMA_VENTANA}")
print(f"   MEJOR_REG_COHORTES  = {MEJOR_REG_COHORTES}")
print(f"   MEJOR_ARIMA_ORDER   = {MEJOR_ARIMA_ORDER}")
print(f"{'='*55}")


# 3. Selección, Aplicación y Guardado de CSVs

In [ ]:
# ─── CONFIGURACIÓN AUTOMÁTICA (desde grid search) ─────────────────────────
# Los parámetros vienen del grid search de la celda anterior.
# Si querés forzar valores manuales, descomentá y cambiá las líneas de abajo.
try:
    METODO_APLICAR        = MEJOR_METODO
    APLICAR_N_VENTANA     = MEJOR_WMA_VENTANA
    APLICAR_N_COHORTES    = MEJOR_REG_COHORTES
    APLICAR_ARIMA_ORDER   = MEJOR_ARIMA_ORDER
except NameError:
    # Fallback si no se corrió el grid search
    print("⚠️ Grid search no ejecutado, usando valores por defecto")
    METODO_APLICAR        = "wma"
    APLICAR_N_VENTANA     = 10
    APLICAR_N_COHORTES    = 10
    APLICAR_ARIMA_ORDER   = (2,1,1)

APLICAR_ARIMA_N       = 12      # ARIMA: ventana de cohortes
APLICAR_N_FUTURAS     = 3       # cohortes futuras a proyectar
# ──────────────────────────────────────────────────────────────────────────
# ──────────────────────────────────────────────────────────────────────────

def _sig_cohorte(c):
    y, m = map(int, c.split("-"))
    m += 1
    if m > 12: m, y = 1, y + 1
    return f"{y:04d}-{m:02d}"

def _estim_reg_full(mob1_series, n_c, n_fut):
    """Devuelve df con historicas + sintéticas para regresion_df."""
    cohs = list(mob1_series.index)
    reg_chs = cohs[-min(n_c, len(cohs)):]
    x_reg = np.arange(len(reg_chs))
    y_reg = mob1_series.loc[reg_chs].values
    coefs = np.polyfit(x_reg, y_reg, 1)
    poly  = np.poly1d(coefs)
    std   = np.std(y_reg - poly(x_reg), ddof=1) if len(y_reg) > 1 else 0.0
    rows = []
    for c in cohs:
        en = c in reg_chs
        idx = reg_chs.index(c) if en else None
        rows.append(dict(cohorte=c, mob1_observado=mob1_series[c],
                         mob1_regresion=poly(idx) if idx is not None else None,
                         mob1_reg_sup=(poly(idx)+std) if idx is not None else None,
                         mob1_reg_inf=(poly(idx)-std) if idx is not None else None,
                         mob1_wma=None, mob1_wma_sup=None, mob1_wma_inf=None,
                         mob1_arima=None, mob1_arima_sup=None, mob1_arima_inf=None,
                         tipo="historica", en_regresion=en))
    ult = cohs[-1]
    for j in range(1, n_fut + 1):
        ult = _sig_cohorte(ult)
        val = poly(len(reg_chs) - 1 + j)
        rows.append(dict(cohorte=ult, mob1_observado=None,
                         mob1_regresion=val, mob1_reg_sup=val+std, mob1_reg_inf=val-std,
                         mob1_wma=None, mob1_wma_sup=None, mob1_wma_inf=None,
                         mob1_arima=None, mob1_arima_sup=None, mob1_arima_inf=None,
                         tipo="sintetica", en_regresion=False))
    return pd.DataFrame(rows), std

def _estim_wma_full(mob1_series, n_v, n_fut):
    cohs = list(mob1_series.index)
    v = mob1_series.iloc[-min(n_v, len(mob1_series)):].values
    w = np.exp(np.linspace(0, 2, len(v))); w /= w.sum()
    wma = np.average(v, weights=w)
    diffs = np.diff(v)
    if len(diffs) > 0:
        wd = np.exp(np.linspace(0, 2, len(diffs))); wd /= wd.sum()
        tend = np.average(diffs, weights=wd)
    else:
        tend = 0.0
    res_wma = []
    for i in range(1, len(v)):
        sub = v[:i]; sw = np.exp(np.linspace(0, 2, len(sub))); sw /= sw.sum()
        res_wma.append(v[i] - np.average(sub, weights=sw))
    std = np.std(res_wma, ddof=1) if len(res_wma) > 1 else 0.0
    rows = [dict(cohorte=c, mob1_observado=mob1_series[c],
                 mob1_regresion=None, mob1_reg_sup=None, mob1_reg_inf=None,
                 mob1_wma=None, mob1_wma_sup=None, mob1_wma_inf=None,
                 mob1_arima=None, mob1_arima_sup=None, mob1_arima_inf=None,
                 tipo="historica", en_regresion=False) for c in cohs]
    ult = cohs[-1]
    for j in range(1, n_fut + 1):
        ult = _sig_cohorte(ult)
        val = wma + tend * j
        rows.append(dict(cohorte=ult, mob1_observado=None,
                         mob1_regresion=None, mob1_reg_sup=None, mob1_reg_inf=None,
                         mob1_wma=val, mob1_wma_sup=val+std, mob1_wma_inf=val-std,
                         mob1_arima=None, mob1_arima_sup=None, mob1_arima_inf=None,
                         tipo="sintetica", en_regresion=False))
    return pd.DataFrame(rows), std

def _estim_arima_full(mob1_series, n_ar, n_fut, order):
    cohs = list(mob1_series.index)
    datos = mob1_series.iloc[-min(n_ar, len(mob1_series)):].values
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            res = _ARIMA(datos, order=order).fit()
            forecast = res.forecast(steps=n_fut)
            std = np.std(res.resid, ddof=1)
        except Exception:
            _, (forecast, std) = None, (_pred_wma(datos, len(datos), n_fut), 0.0)
    rows = [dict(cohorte=c, mob1_observado=mob1_series[c],
                 mob1_regresion=None, mob1_reg_sup=None, mob1_reg_inf=None,
                 mob1_wma=None, mob1_wma_sup=None, mob1_wma_inf=None,
                 mob1_arima=None, mob1_arima_sup=None, mob1_arima_inf=None,
                 tipo="historica", en_regresion=False) for c in cohs]
    ult = cohs[-1]
    for j, val in enumerate(forecast):
        ult = _sig_cohorte(ult)
        rows.append(dict(cohorte=ult, mob1_observado=None,
                         mob1_regresion=None, mob1_reg_sup=None, mob1_reg_inf=None,
                         mob1_wma=None, mob1_wma_sup=None, mob1_wma_inf=None,
                         mob1_arima=val, mob1_arima_sup=val+std, mob1_arima_inf=val-std,
                         tipo="sintetica", en_regresion=False))
    return pd.DataFrame(rows), std

def _construir_curva(mob1_val, factores_cl, mob_maduro):
    mob1_val = max(0.001, min(1.0, mob1_val))
    vals = {"MOB_1": mob1_val}
    cur = mob1_val
    for mn in range(1, mob_maduro):
        trans = f"{mn}->{mn+1}"
        if trans in factores_cl:
            cur = cur * factores_cl[trans]
            vals[f"MOB_{mn+1}"] = cur
        else:
            break
    return vals

# ─── Cargar factores CL ───────────────────────────────────────────────────
factores_df = pd.read_csv(
    os.path.join(PROCESSED_DIR, "factores_cl_niveles.csv"), sep=";", decimal=","
)

col_mob1_map = {"wma": "mob1_wma", "regresion": "mob1_regresion", "arima": "mob1_arima"}
col_mob1_usar = col_mob1_map[METODO_APLICAR]

all_sint, all_reg, all_sint_dict = [], [], {}

for seg in sorted(matrices_obs["segmento"].unique()):
    sub = matrices_obs[matrices_obs["segmento"] == seg].copy()
    if "MOB_1" not in sub.columns:
        continue
    mob1 = sub.set_index("cohorte")["MOB_1"].dropna().sort_index()
    cohs_ord = list(mob1.index)

    # Factores CL del segmento
    row_f = factores_df[factores_df["segmento"] == seg]
    fcl = {}
    if not row_f.empty:
        for col in row_f.columns:
            if "->" in col:
                v = row_f[col].iloc[0]
                if pd.notna(v):
                    fcl[col] = float(v)

    # Segmentos deshabilitados → ceros
    if seg in SEGMENTOS_SKIP:
        ult = cohs_ord[-1]
        filas_s, filas_r = [], []
        for j in range(1, APLICAR_N_FUTURAS + 1):
            ult = _sig_cohorte(ult)
            fs = {"cohorte": ult, "segmento": seg}
            for m in range(1, MOB_MADURO + 1): fs[f"MOB_{m}"] = 0.0
            filas_s.append(fs)
            filas_r.append(dict(cohorte=ult, mob1_observado=None,
                                mob1_regresion=0.0, mob1_reg_sup=0.0, mob1_reg_inf=0.0,
                                mob1_wma=0.0, mob1_wma_sup=0.0, mob1_wma_inf=0.0,
                                mob1_arima=0.0, mob1_arima_sup=0.0, mob1_arima_inf=0.0,
                                tipo="sintetica", en_regresion=False, segmento=seg))
        for c in cohs_ord:
            filas_r.insert(0, dict(cohorte=c, mob1_observado=mob1[c],
                                   mob1_regresion=None, mob1_reg_sup=None, mob1_reg_inf=None,
                                   mob1_wma=None, mob1_wma_sup=None, mob1_wma_inf=None,
                                   mob1_arima=None, mob1_arima_sup=None, mob1_arima_inf=None,
                                   tipo="historica", en_regresion=False, segmento=seg))
        df_s = pd.DataFrame(filas_s).set_index("cohorte")
        all_sint.append(df_s.reset_index())
        all_sint_dict[seg] = df_s.drop(columns=["segmento"])
        all_reg.append(pd.DataFrame(filas_r))
        continue

    if mob1.shape[0] < 4:
        continue

    # Estimar según método elegido
    if METODO_APLICAR == "regresion":
        df_r, _ = _estim_reg_full(mob1, APLICAR_N_COHORTES, APLICAR_N_FUTURAS)
    elif METODO_APLICAR == "wma":
        df_r, _ = _estim_wma_full(mob1, APLICAR_N_VENTANA, APLICAR_N_FUTURAS)
    else:
        df_r, _ = _estim_arima_full(mob1, APLICAR_ARIMA_N, APLICAR_N_FUTURAS, APLICAR_ARIMA_ORDER)

    df_r["segmento"] = seg
    all_reg.append(df_r)

    # Construir curvas sintéticas completas
    sint_rows = []
    for _, row in df_r[df_r["tipo"] == "sintetica"].iterrows():
        mob1_val = row[col_mob1_usar]
        if pd.isna(mob1_val):
            continue
        curva = _construir_curva(mob1_val, fcl, MOB_MADURO)
        curva["cohorte"] = row["cohorte"]
        curva["segmento"] = seg
        sint_rows.append(curva)

    if sint_rows:
        df_s = pd.DataFrame(sint_rows).set_index("cohorte")
        all_sint.append(df_s.reset_index())
        all_sint_dict[seg] = df_s.drop(columns=["segmento"])

# ─── Actualizar DataFrames globales ──────────────────────────────────────
regresion_df  = pd.concat(all_reg, ignore_index=True)
sinteticas_df = pd.concat(all_sint, ignore_index=True)

mob_cols_s = (["segmento", "cohorte"] +
              [f"MOB_{m}" for m in range(1, MOB_MADURO + 1)
               if f"MOB_{m}" in sinteticas_df.columns])
sinteticas_df = sinteticas_df[[c for c in mob_cols_s if c in sinteticas_df.columns]]

# General ponderada
mob1_data = df_consolidado[df_consolidado["mob"] == 1]
pesos = mob1_data.pivot_table(index="cohorte", columns="segmento",
                               values="cantidad_operaciones", aggfunc="sum")
cohortes_sint_futuras = sorted(sinteticas_df["cohorte"].unique())
mob_cols_gen = [f"MOB_{m}" for m in range(1, MOB_MADURO + 1)]
general_nuevo = pd.DataFrame(index=cohortes_sint_futuras, columns=mob_cols_gen, dtype=float)

for cohorte in cohortes_sint_futuras:
    for col in mob_cols_gen:
        sp, sw = 0.0, 0.0
        for seg, sdf in all_sint_dict.items():
            if cohorte not in sdf.index or col not in sdf.columns:
                continue
            val = sdf.loc[cohorte, col]
            if pd.isna(val):
                continue
            if seg in pesos.columns:
                p = pesos[seg].dropna()
                peso = p.iloc[-3:].mean() if len(p) >= 3 else p.mean()
            else:
                peso = 0
            if peso > 0:
                sp += val * peso; sw += peso
        if sw > 0:
            general_nuevo.loc[cohorte, col] = sp / sw

general_nuevo.index.name = "cohorte"
general_sint    = general_nuevo.copy()
general_futuras = general_sint.loc[general_sint.index > ultima_hist].copy()

print("=" * 60)
print(f"Configuración aplicada: MÉTODO = {METODO_APLICAR.upper()}")
if METODO_APLICAR == "wma":
    print(f"  WMA  → n_ventana   = {APLICAR_N_VENTANA}")
elif METODO_APLICAR == "regresion":
    print(f"  Reg  → n_cohortes  = {APLICAR_N_COHORTES}")
else:
    print(f"  ARIMA order={APLICAR_ARIMA_ORDER}  n={APLICAR_ARIMA_N}")
print(f"  Cohortes futuras  : {cohortes_sint_futuras}")
print(f"  Segmentos activos : {sinteticas_df['segmento'].nunique()}")
print("  → Sección 3 en adelante usa esta configuración.")
print("=" * 60)


# ─── Guardar CSVs actualizados ────────────────────────────────────────────
general_nuevo.to_csv(
    os.path.join(PROCESSED_DIR, "sinteticas_general_niveles.csv"),
    sep=";", decimal=","
)
sinteticas_df.to_csv(
    os.path.join(PROCESSED_DIR, "sinteticas_niveles.csv"),
    sep=";", decimal=",", index=False
)
regresion_df.to_csv(
    os.path.join(PROCESSED_DIR, "regresion_mob1_niveles.csv"),
    sep=";", decimal=",", index=False
)
print("\n✅ CSVs guardados en data/processed/ con la configuración óptima del grid search")
print("   → sinteticas_general_niveles.csv")
print("   → sinteticas_niveles.csv")
print("   → regresion_mob1_niveles.csv")
print("   El tablero_resumen_ejecutivo.ipynb usará estos valores actualizados.")


# 4. Visualización de Resultados Integrados

---
## 3. Curvas Sinteticas vs Historicas por Segmento (12 graficos)

Curvas historicas en azul/naranja, sinteticas en **verde punteado**.

In [ ]:
def get_matriz(df, segmento):
    sub = df[df["segmento"] == segmento].copy()
    mob_cols = [c for c in sub.columns if c.startswith("MOB_")]
    return sub.set_index("cohorte")[mob_cols]

def extraer_serie(matriz, cohorte):
    vals = matriz.loc[cohorte].dropna()
    mobs = [int(c.replace("MOB_", "")) for c in vals.index]
    return mobs, vals.values

fig, axes = plt.subplots(4, 3, figsize=(22, 24))
fig.suptitle("Curvas Sinteticas vs Historicas por Segmento",
             fontsize=16, fontweight="bold", y=0.98)

for idx, segmento in enumerate(segmentos):
    ax = axes[idx // 3, idx % 3]
    m_obs = get_matriz(matrices_obs, segmento)

    # Historicas
    for cohorte in sorted(m_obs.index):
        mobs, vals = extraer_serie(m_obs, cohorte)
        if cohorte.startswith("2024"):
            color = COLOR_2024
        elif cohorte.startswith("2025"):
            color = COLOR_2025
        else:
            color = COLOR_2026
        ax.plot(mobs, vals, color=color, alpha=ALPHA_INDIVIDUAL,
                linewidth=1.0, zorder=1)

    # Sinteticas
    m_sint = get_matriz(sinteticas_df, segmento) if segmento in sinteticas_df["segmento"].values else None
    if m_sint is not None and not m_sint.empty:
        for cohorte in sorted(m_sint.index):
            if cohorte <= ultima_hist:
                continue  # solo futuras
            mobs, vals = extraer_serie(m_sint, cohorte)
            ax.plot(mobs, vals, color=COLOR_SINT, linewidth=2.0,
                    linestyle="--", alpha=0.8, zorder=3)
            ax.annotate(cohorte, xy=(mobs[-1], vals[-1]),
                        xytext=(4, 0), textcoords="offset points",
                        fontsize=7, fontweight="bold", color=COLOR_SINT, va="center")

    partes = segmento.split("_")
    tipo = NOMBRE_TIPOOPE.get(partes[0], partes[0])
    cliente = NOMBRE_CLIENTENUEVO.get(partes[1], partes[1])
    nivel = partes[2]
    ax.set_title(f"{tipo} - {cliente} - {nivel}",
                 fontsize=11, fontweight="bold", color=COLOR_NIVEL.get(nivel, "black"))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    ax.grid(True, alpha=0.25, linestyle="--")
    ax.set_xlabel("MOB", fontsize=9)
    ax.set_ylabel("Indice Morosidad", fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(REPORTS_DIR, "curvas_sinteticas_niveles_12.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Guardado en reports/curvas_sinteticas_niveles_12.png")

---
## 4. Detalle MOB a MOB de Cohortes Sinteticas por Segmento

In [ ]:
print("DETALLE MOB A MOB - COHORTES SINTETICAS POR SEGMENTO")
print("=" * 100)

for segmento in segmentos:
    m_sint = get_matriz(sinteticas_df, segmento) if segmento in sinteticas_df["segmento"].values else None
    if m_sint is None or m_sint.empty:
        continue

    # Solo futuras
    futuras = [c for c in sorted(m_sint.index) if c > ultima_hist]
    if not futuras:
        continue

    partes = segmento.split("_")
    tipo = NOMBRE_TIPOOPE.get(partes[0], partes[0])
    cliente = NOMBRE_CLIENTENUEVO.get(partes[1], partes[1])
    nivel = partes[2]

    print(f"\n--- {tipo} - {cliente} - {nivel} ---")
    mob_cols = [c for c in m_sint.columns if c.startswith("MOB_")]
    mob_cols = sorted(mob_cols, key=lambda x: int(x.replace("MOB_", "")))

    header = f"{'Cohorte':>10s} |"
    for col in mob_cols:
        header += f" {col.replace('MOB_','M'):>6s}"
    print(header)
    print("-" * len(header))

    for cohorte in futuras:
        line = f"{cohorte:>10s} |"
        for col in mob_cols:
            val = m_sint.loc[cohorte, col] if col in m_sint.columns else np.nan
            if pd.notna(val):
                line += f" {val:>5.1%}"
            else:
                line += f" {'---':>6s}"
        print(line)

print("\n" + "=" * 100)

---
## 5. Proyeccion General Ponderada - Sinteticas

Curva general que pondera los 12 segmentos por operaciones. Solo cohortes verdaderamente futuras.

In [ ]:
# Leer matrices historicas observadas para superponer
matrices_proy = pd.read_csv(os.path.join(PROCESSED_DIR, "matrices_proyectadas_niveles.csv"), sep=";", decimal=",")

# Reconstruir general historica (observada + CL)
general_proy = pd.read_csv(os.path.join(PROCESSED_DIR, "proyeccion_general_niveles.csv"), sep=";", decimal=",", index_col=0)

fig, ax = plt.subplots(figsize=(16, 8))

# Historicas (semitransparentes)
for cohorte in sorted(general_proy.index):
    vals = general_proy.loc[cohorte].dropna()
    mobs = [int(c.replace("MOB_", "")) for c in vals.index]
    if cohorte.startswith("2024"):
        color = COLOR_2024
    elif cohorte.startswith("2025"):
        color = COLOR_2025
    else:
        color = COLOR_2026
    ax.plot(mobs, vals.values, color=color, alpha=ALPHA_INDIVIDUAL,
            linewidth=1.0, zorder=1)

# Sinteticas futuras (verdes punteadas, gruesas)
for cohorte in general_futuras.index:
    vals = general_futuras.loc[cohorte].dropna()
    mobs = [int(c.replace("MOB_", "")) for c in vals.index]
    ax.plot(mobs, vals.values, color=COLOR_SINT, linewidth=2.5,
            linestyle="--", alpha=0.85, zorder=3)
    ax.annotate(cohorte, xy=(mobs[-1], vals.iloc[-1]),
                xytext=(6, 0), textcoords="offset points",
                fontsize=9, fontweight="bold", color=COLOR_SINT, va="center", zorder=5)
    ax.plot(mobs[-1], vals.iloc[-1], "o", color=COLOR_SINT, markersize=6, zorder=5)

ax.set_xlabel("Months on Books (MOB)", fontsize=12)
ax.set_ylabel("Indice de Morosidad", fontsize=12)
ax.set_title("Cohortes Sinteticas - Proyeccion General Ponderada",
             fontsize=14, fontweight="bold")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_xticks(range(1, MOB_MADURO + 1))
ax.grid(True, alpha=0.25, linestyle="--")

legend_elements = [
    Line2D([0], [0], color=COLOR_2024, alpha=ALPHA_INDIVIDUAL, lw=4, label="Historicas 2024"),
    Line2D([0], [0], color=COLOR_2025, alpha=ALPHA_INDIVIDUAL, lw=4, label="Historicas 2025"),
    Line2D([0], [0], color=COLOR_SINT, lw=2.5, ls="--", label="Sinteticas (futuras)"),
]
ax.legend(handles=legend_elements, fontsize=10, loc="upper left", framealpha=0.9)

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, "sinteticas_general_niveles.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Guardado en reports/sinteticas_general_niveles.png")

### Tabla detallada - Proyeccion General Ponderada (cohortes futuras)

In [ ]:
mob_cols = [f"MOB_{m}" for m in range(1, MOB_MADURO + 1)]
mob_cols_disp = [c for c in mob_cols if c in general_futuras.columns]

print("DETALLE MOB A MOB - COHORTES SINTETICAS GENERAL PONDERADA")
print("=" * 110)
header = f"{'Cohorte':>10s} |"
for col in mob_cols_disp:
    header += f" {col.replace('MOB_','M'):>6s}"
header += f" | {'M.Ultima':>8s}"
print(header)
print("-" * 110)

for cohorte in sorted(general_futuras.index):
    line = f"{cohorte:>10s} |"
    for col in mob_cols_disp:
        val = general_futuras.loc[cohorte, col]
        if pd.notna(val):
            line += f" {val:>5.1%}"
        else:
            line += f" {'---':>6s}"
    # Mora ultima
    last_val = general_futuras.loc[cohorte].dropna().iloc[-1] if general_futuras.loc[cohorte].notna().any() else np.nan
    line += f" | {last_val:>7.2%}" if pd.notna(last_val) else f" | {'---':>8s}"
    print(line)

print("=" * 110)

# Tabla estilizada
general_futuras[mob_cols_disp].style.format("{:.2%}", na_rep="-").background_gradient(
    cmap="RdYlGn_r", axis=None
).set_caption("Cohortes Sinteticas - Proyeccion General Ponderada")

---
## 6. Comparacion Mora Ultima: Sinteticas vs Historicas por Nivel

In [ ]:
mob_col_maduro = f"MOB_{MOB_MADURO}"

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f"Mora al MOB {MOB_MADURO}: Historicas vs Sinteticas por Nivel",
             fontsize=14, fontweight="bold", y=0.98)

for idx, (tipoope, clientenuevo) in enumerate(combos):
    ax = axes[idx // 2, idx % 2]
    tipo_n = NOMBRE_TIPOOPE[tipoope]
    cliente_n = NOMBRE_CLIENTENUEVO[clientenuevo]

    for nivel in niveles:
        seg = f"{tipoope}_{clientenuevo}_{nivel}"

        # Historicas
        m_obs = get_matriz(matrices_obs, seg)
        if mob_col_maduro in m_obs.columns:
            hist_vals = m_obs[mob_col_maduro].dropna()
        else:
            hist_vals = pd.Series(dtype=float)

        # Sinteticas
        m_sint = get_matriz(sinteticas_df, seg) if seg in sinteticas_df["segmento"].values else None
        if m_sint is not None and mob_col_maduro in m_sint.columns:
            sint_vals = m_sint.loc[[c for c in m_sint.index if c > ultima_hist], mob_col_maduro].dropna()
        else:
            sint_vals = pd.Series(dtype=float)

        # Plotear historicas como linea
        if not hist_vals.empty:
            ax.plot(range(len(hist_vals)), hist_vals.values,
                    color=COLOR_NIVEL[nivel], alpha=0.6, linewidth=1.5,
                    marker="o", markersize=4, label=f"{nivel} hist (n={len(hist_vals)})")

        # Sinteticas como cuadrados
        if not sint_vals.empty:
            x_sint = range(len(hist_vals), len(hist_vals) + len(sint_vals))
            ax.plot(x_sint, sint_vals.values,
                    color=COLOR_NIVEL[nivel], linewidth=2, linestyle="--",
                    marker="s", markersize=6, alpha=0.9)

    ax.set_title(f"{tipo_n} - {cliente_n}", fontsize=12, fontweight="bold")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    ax.grid(True, alpha=0.25, linestyle="--")
    ax.legend(fontsize=8)
    ax.set_xlabel("Cohorte (indice)", fontsize=9)
    ax.set_ylabel(f"Indice al MOB {MOB_MADURO}", fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(REPORTS_DIR, "sinteticas_vs_historicas_niveles.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Guardado en reports/sinteticas_vs_historicas_niveles.png")

---
## 7. Resumen Ejecutivo

In [ ]:
mob_col_maduro = f"MOB_{MOB_MADURO}"

print("=" * 70)
print("RESUMEN - COHORTES SINTETICAS POR NIVELES DE RIESGO")
print("=" * 70)

# General
print(f"\nPROYECCION GENERAL PONDERADA (cohortes futuras):")
for cohorte in sorted(general_futuras.index):
    vals = general_futuras.loc[cohorte].dropna()
    mob1 = vals.iloc[0] if len(vals) > 0 else np.nan
    ultima = vals.iloc[-1] if len(vals) > 0 else np.nan
    ultimo_mob = vals.index[-1] if len(vals) > 0 else "-"
    print(f"  {cohorte}: MOB_1={mob1:.2%} -> {ultimo_mob}={ultima:.2%}")

# Por nivel
print(f"\nMORA AL MOB {MOB_MADURO} - SINTETICAS POR NIVEL:")
print("-" * 60)

for nivel in niveles:
    segs = [s for s in segmentos if s.endswith(f"_{nivel}")]
    vals_sint = []
    for seg in segs:
        m_sint = get_matriz(sinteticas_df, seg) if seg in sinteticas_df["segmento"].values else None
        if m_sint is not None and mob_col_maduro in m_sint.columns:
            futuras = m_sint.loc[[c for c in m_sint.index if c > ultima_hist], mob_col_maduro].dropna()
            vals_sint.extend(futuras.tolist())
    if vals_sint:
        print(f"  {nivel:6s}: promedio={np.mean(vals_sint):.2%}, "
              f"rango=[{np.min(vals_sint):.2%}, {np.max(vals_sint):.2%}], n={len(vals_sint)}")

# Por segmento
print(f"\nPOR SEGMENTO (promedio sinteticas futuras):")
print("-" * 70)
seg_data = []
for seg in segmentos:
    m_sint = get_matriz(sinteticas_df, seg) if seg in sinteticas_df["segmento"].values else None
    if m_sint is None:
        continue
    futuras_idx = [c for c in m_sint.index if c > ultima_hist]
    if not futuras_idx:
        continue
    mob1_prom = m_sint.loc[futuras_idx, "MOB_1"].mean() if "MOB_1" in m_sint.columns else np.nan
    ult_prom = m_sint.loc[futuras_idx, mob_col_maduro].mean() if mob_col_maduro in m_sint.columns else np.nan
    seg_data.append((seg, mob1_prom, ult_prom, len(futuras_idx)))

seg_data.sort(key=lambda x: x[2] if pd.notna(x[2]) else 0, reverse=True)
for seg, mob1, ult, n in seg_data:
    partes = seg.split("_")
    tipo = NOMBRE_TIPOOPE.get(partes[0], partes[0])
    cliente = NOMBRE_CLIENTENUEVO.get(partes[1], partes[1])
    nivel = partes[2]
    nombre = f"{tipo} {cliente} {nivel}"
    mob1_s = f"{mob1:.2%}" if pd.notna(mob1) else "-"
    ult_s = f"{ult:.2%}" if pd.notna(ult) else "-"
    print(f"  {nombre:25s}: MOB_1={mob1_s:>7s} -> MOB_{MOB_MADURO}={ult_s:>7s} (n={n})")

print("\n" + "=" * 70)